# Lab 10 · Routing: rules, models, and a safe fallback

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Route resilience questions by uncertainty and risk
**Start with a problem:** Sending every question to a full review can cost time, while an automatic decision can miss an unsafe payment.

**In this lab:** Route clear cases to a simple rule and uncertain or risky cases to review.

**So far:** the human approved retries only for safe effects. **New:** a model router chooses a cheap rule or a careful review queue. ShopEasy's checkout depends on Pricing, Inventory, Payments, and Recommendations.

The rule gives narrow advice for known idempotent or non-idempotent effects. A case with unknown idempotency or a business consequence that needs judgement goes to the council/human queue. Invalid model output also falls back to review. This lab recommends policies; it sends no requests and charges no cards.

**At a glance**

![Lab 10 at a glance: Case (facts + note), then Router (route + reason), then valid?, then Safe rule (0 extra model calls) or Review (uncertain effects), then Compare (vs human routes).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/10.svg)

## Think about the example

A Pricing lookup is a read. A Payments charge can change a customer's balance even if the response times out. A router itself can add model calls before a case reaches review.

**Concepts:** Retry safety and fallback choices; Different costs of routing errors.

Use the scenario above to consider these questions before running the code:

1. Why is the Payments case more dangerous than a failed Pricing read?
2. Why can routing cost as much as sending every case to review?

## The idea in context

Routing chooses which workflow handles a case. A dispatch table and conservative fallback keep selection separate from execution.

A router spends calls too; an uncertain reply should not silently skip review.

| Key term | Meaning |
| --- | --- |
| **Router** | A component that picks the handler for each input. |
| **Dispatch table** | A mapping from each route to the function that handles it. |
| **Fallback** | The safe route taken when the router's answer cannot be used. |
| **Asymmetric errors** | Mistakes whose costs differ, so they are counted and weighed separately. |

## Pattern · Routing

Routing is not a separate entry in the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. It combines two of the guide's ideas: the risk classification of *Human in the Loop* (classify, then send only high-risk cases to the expensive path) and the dispatch of *Registry & Adapter* (look up a handler and call it through one interface).

| | |
| --- | --- |
| **Context** | Cases differ in how much deliberation they need, and the most careful path is also the most expensive. |
| **Problem** | Sending every case to the council wastes calls; sending every case to a rule skips debates that were needed. |
| **Solution** | Classify each case, dispatch it to the handler that fits, and fall back to the careful path when the classification is unusable. |
| **Use when** | Cases fall into recognizable classes; handlers differ greatly in cost; one kind of wrong route is cheaper than the other. |
| **Trade-off** | Saves calls on routine cases, but the router adds its own calls, and its mistakes stay silent unless they are measured. |
| **Flow** | Case → Classify → Dispatch → Handler |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 12 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/5afd3e9402e35da6cba234dd9e794a387595c5c6.zip"
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Step 1 · Inputs and human routes

These short notes are teaching scenarios created for the exercise, not measured service behaviour. The held-out route labels are used for evaluation, never included in the router's prompt.

In [ ]:
import json

cases = [
    {"name": "Pricing", "idempotent": True, "note": "Transient quote-lookup timeout; bounded retry fits the latency budget."},
    {"name": "Inventory", "idempotent": None, "note": "A reservation timed out; its contract does not state idempotency."},
    {"name": "Payments", "idempotent": False, "note": "The charge outcome is uncertain; reconcile status before another charge."},
    {"name": "Recommendations", "idempotent": True, "note": "Optional and flaky; decide whether fallback is better than more waiting."},
    {"name": "Catalog", "idempotent": True, "note": "Transient product-lookup timeout; bounded retry fits the latency budget."},
]
human_routes = {"Pricing": "rule", "Inventory": "review", "Payments": "review",
                "Recommendations": "review", "Catalog": "rule"}

## Step 2 · The program owns handlers

`apply_rule` never automatically retries a known non-idempotent effect. The review handler only queues a name. Its estimated five model calls are an accounting assumption, not an executed debate or a fixed cost of every council.

In [ ]:
POLICY = "Retry only idempotent effects, bounded with exponential backoff and jitter. Never blindly repeat a payment charge."
COST = {"rule": 0, "review": 5}  # illustrative future review cost, not measured here
review_queue = []

def apply_rule(case):
    if case["idempotent"] is True:
        return "bounded retry with backoff and jitter"
    return "do not automatically retry"

def send_to_review(case):
    review_queue.append(case["name"])
    return "queued for council/human review"

HANDLERS = {"rule": apply_rule, "review": send_to_review}
print(apply_rule(cases[2]))  # even a misroute must not repeat the non-idempotent payment

## Step 3 · Route with a cautious fallback

The model weighs the free-text note. Known routine read failures use the rule; unknown contracts, uncertain payment outcomes, and fallback decisions need review. A malformed or unknown route becomes `review`. The deterministic handlers remain in charge of advice and execution boundaries.

The router returns a route and a reason. A malformed reply or unknown route goes to review, where a person can inspect the case.

In [ ]:
EXAMPLES = ("Transient idempotent lookup, latency budget available -> rule. "
            "Unknown effect idempotency -> review. Uncertain non-idempotent charge -> review. "
            "Optional service needs fallback judgement -> review.")

def route(case):
    prompt = (f"Policy: {POLICY}\n{EXAMPLES}\nCase: {json.dumps(case)}\n"
              'Reply with JSON only: {"route": "rule" or "review", "reason": "one sentence"}')
    try:
        reply = json.loads(ask(prompt))
    except json.JSONDecodeError:
        return "review", "not JSON"
    if not isinstance(reply, dict) or reply.get("route") not in HANDLERS:
        return "review", "unknown or malformed route"
    return reply["route"], reply.get("reason", "")

## Step 4 · Count both kinds of mistake

A wasted review adds calls. A skipped review bypasses necessary judgement. Count the router's calls too. The budget below estimates queued debates; only the routing calls actually run.

A skipped review can miss a risky case; a wasted review uses extra effort. Both are shown separately.

In [ ]:
calls, wasted, skipped = len(cases), [], []
for case in cases:
    chosen, reason = route(case)
    result = HANDLERS[chosen](case)
    calls += COST[chosen]
    if chosen != human_routes[case["name"]]:
        if chosen == "rule":
            skipped.append(case["name"])
        else:
            wasted.append(case["name"])
    print(case["name"], "->", chosen, "|", result, "|", reason)
print("Agreed:", len(cases) - len(wasted) - len(skipped), "/", len(cases))
print("Estimated calls including queued reviews:", calls, "| all-review:", COST["review"] * len(cases))
print("Wasted reviews:", wasted, "| skipped reviews:", skipped)
assert not skipped, "A case requiring judgement skipped review."

## What you should see

The router selects a handler for each case. Compare the routes with the human labels, and inspect both missed reviews and unnecessary reviews. Call counts are estimates and depend on the model’s choices.

## Try it

1. Route Payments to `rule`. The evaluation flags the skipped review; the handler still says not to retry automatically.
2. Make a reply `[]` or give it the route `maybe`. Both fall back to review.
3. Add a timeout budget to Recommendations. When should a fallback replace another attempt?
4. Replace Inventory's unknown idempotency with a documented idempotency key and a reconciliation API. Reconsider its route with a domain expert.

## Takeaway

A router allocates effort; it does not create safety by itself. Deterministic handlers retain the no-blind-payment-retry rule, unknown cases get reviewed, and measured routing errors reveal whether the extra model call was worthwhile.